# Flow Matching / 扩散模型「原理篇」· 核验 notebook

配套博文：**《从概率路径到 Flow Matching：用一套数学框架理解现代扩散模型》**
<https://lrypcy.github.io/2026/10/20/flow-matching-00-principles/>

底本：Peter Holderrieth & Ezra Erives, *An Introduction to Flow Matching and
Diffusion Models*, MIT 6.S184 (2026) — <https://diffusion.csail.mit.edu/>

**这个 notebook 在做什么。** 把博文里每一条“看起来显然”的恒等式都拿数值算一遍，
让它们的误差落到机器精度，而不是停在“推一遍就相信”。

**为什么残差能做到 $10^{-13}$ 而不是 $10^{-10}$。** 关键在选择 $p_{\rm data}$：
这里把它取成**有限个原子的加权混合**。于是 $p_t$、$\nabla\log p_t$、$u_t$、
$\mathrm{div}(u_t)$、$\Delta p_t$ 全部有解析闭式，偏微分方程的残差不需要任何
数值差分——差分格式本身的截断误差被彻底绕开。

依赖：numpy（纯 CPU，约 2 分钟跑完）。每个 section 末尾自断言，最后汇总。

In [1]:
import fm_principles_lab as L
import numpy as np

rng = np.random.default_rng(20261020)
print("numpy", np.__version__)
print("组：A 闭式三兄弟 / B CE+FPE / C Thm12 / D 四参数化 / E Thm17 / F CFG / G 端到端训练")

numpy 2.1.1
组：A 闭式三兄弟 / B CE+FPE / C Thm12 / D 四参数化 / E Thm17 / F CFG / G 端到端训练


## A. 高斯概率路径的三张面孔

讲义 Example 10 给出条件向量场的闭式

$$u_t(x\mid z)=\Big(\dot\alpha_t-\frac{\dot\beta_t}{\beta_t}\alpha_t\Big)z+\frac{\dot\beta_t}{\beta_t}x,$$

它由条件流 $\psi_t(x\mid z)=\alpha_t z+\beta_t x$ 反解得到。Prop 1 再说它等于 score 的仿射组合

$$u_t(x)=a_t\nabla\log p_t(x)+b_tx,\qquad a_t=\beta_t^2\Big(\frac{\dot\alpha_t}{\alpha_t}-\frac{\dot\beta_t}{\beta_t}\Big),\quad b_t=\frac{\dot\alpha_t}{\alpha_t},$$

denoiser（后验均值 $\mathbb E[z\mid x]$）则是同一件事的第三种写法（Eq. 43）。

**要验的是**：这三件事 *逐点* 相等，而不是“理论上等价”。

In [2]:
n_ok = L.exp_A(rng)

A | 高斯概率路径：向量场 / score / denoiser 是同一个后验均值的仿射重组
    [ok]   OT(σ_min=0)    条件场 ⇔ 条件流反解 2.84e-14
    [ok]   OT(σ_min=0)    Prop 1  u = a·s + b·x 4.26e-14
    [ok]   OT(σ_min=0)    denoiser 反解 Eq.43 1.11e-15
    [ok]   OT(σ_min=0.05) 条件场 ⇔ 条件流反解 1.42e-14
    [ok]   OT(σ_min=0.05) Prop 1  u = a·s + b·x 3.20e-14
    [ok]   OT(σ_min=0.05) denoiser 反解 Eq.43 8.88e-16
    [ok]   OT(σ_min=0.2)  条件场 ⇔ 条件流反解 3.55e-15
    [ok]   OT(σ_min=0.2)  Prop 1  u = a·s + b·x 2.13e-14
    [ok]   OT(σ_min=0.2)  denoiser 反解 Eq.43 1.55e-15
    [ok]   trig           条件场 ⇔ 条件流反解 4.26e-14
    [ok]   trig           Prop 1  u = a·s + b·x 2.84e-14
    [ok]   trig           denoiser 反解 Eq.43 1.11e-15
    [ok]   poly           条件场 ⇔ 条件流反解 2.84e-14
    [ok]   poly           Prop 1  u = a·s + b·x 4.90e-14
    [ok]   poly           denoiser 反解 Eq.43 1.33e-15

  表 A｜三条恒等式在 5 条路径上的最大绝对误差
  路径                       条件场⇔流        Prop 1      denoiser
  OT(σ_min=0)           2.84e-14      4.26e-14      1.11e-15
  OT(σ_min=0.05) 

## B. 连续性方程与 Fokker–Planck：分布层面的契约

Thm 11（连续性方程）与 Thm 19（Fokker–Planck）回答同一个问题：
**什么样的动力学能配得上这条概率路径**。

$$\partial_t p_t=-\mathrm{div}(p_tu_t),\qquad
\partial_t p_t=-\mathrm{div}(p_tu_t)+\frac{\sigma_t^2}{2}\Delta p_t .$$

注意第二个式子：**它会对「加了噪声的同一个 $u_t$」判死刑**。
因为 $\partial_t p_t=-\mathrm{div}(p_tu_t)$ 已经由连续性方程钉死，
只要 $\sigma_t\neq0$，右端多出来的 $\frac{\sigma^2}{2}\Delta p_t$ 就必须由漂移 absorbed 掉。
这就是 Thm 17 那个奇怪的修正项 $\frac{\sigma_t^2}{2}\nabla\log p_t$ 的来源——它不是技巧，是必需品。

下表把这个“必需品”的成本直接量化出来。

In [3]:
n_ok = L.exp_B(rng)

B | 连续性方程（Thm 11）与 Fokker-Planck（Thm 19）：路径无关的分布契约



  表 B1｜连续性方程相对残差  max|∂_t p + div(p·u)| / max|∂_t p|
  路径                          相对残差
  OT(σ_min=0)             4.13e-15
  OT(σ_min=0.05)          4.80e-15
  OT(σ_min=0.2)           5.98e-15
  trig                    5.20e-15
  poly                    5.34e-15
    [ok]   5 条路径的 CE 残差 < 1e-10 worst 5.98e-15

  表 B2｜同一个 u_t 直接配噪声（不加修正项）会发生什么
       σ_t        FPE 残差 (裸 u_t)       FPE 残差 (u + σ²/2·s)
       0.0              2.36e-14                  2.36e-14
       0.3              3.42e-01                  2.60e-14


       0.8              2.43e+00                  5.36e-14
       2.0              1.52e+01                  2.64e-13
    [ok]   裸 u_t + 噪声会破坏边际分布（σ=0.8 残差 > 0.1） worst 1.52e+01
    [ok]   加上 σ²/2·∇log p_t 后残差回到机器精度 worst 2.64e-13

  ⇒ Thm 17 的修正项不是「可选的技巧」：没有它就配不上同一条 p_t。



## C. Thm 12：为什么训练目标可以换成条件版的 MSE

flow matching 的全部可扩展性都压在这条定理上：

$$\mathcal L_{\rm FM}(\theta)=\mathcal L_{\rm CFM}(\theta)+C,\qquad C\ \text{与}\ \theta\ \text{无关}.$$

证明只用到「平方展开 + 期望线性化」，没有任何近似。
**但这条定理特别容易被“验错”**：$\mathcal L_{\rm FM}$ 里出现的是边际向量场，
它的重尾（CondOT 下含 $1/(1-t)$ 因子）会让蒙特卡洛估计量的方差大到吞掉常数本身。

判据很简单：**残差必须随网格/样本量收敛**，不收敛就说明是估计量不可用，不是定理错。

In [4]:
n_ok = L.exp_C(rng)

C | Thm 12：∇_θ L_FM = ∇_θ L_CFM —— 以及为什么不能用蒙特卡洛验它

  (1) 确定性求积：差值在整个 θ 网格上是常数
  路径                  t            差值 C         跨 θ 的跨度          相对跨度
  OT(σ_min=0)       0.2      5.14430422        2.66e-15      5.18e-16
  OT(σ_min=0)       0.5      2.29153579        1.78e-15      7.75e-16
  OT(σ_min=0)       0.8      0.00180236        1.78e-15      9.86e-13
  OT(σ_min=0.05)    0.2      5.04544463        3.55e-15      7.04e-16
  OT(σ_min=0.05)    0.5      2.33013769        3.55e-15      1.52e-15
  OT(σ_min=0.05)    0.8      0.01686636        5.33e-15      3.16e-13
  trig              0.2      7.76882070        4.44e-15      5.72e-16


  trig              0.5      2.82706897        2.66e-15      9.43e-16
  trig              0.8      0.06085767        1.33e-15      2.19e-14
    [ok]   求积下 L_FM − L_CFM 与 θ 无关（相对跨度 < 1e-8） worst rel 9.86e-13

  (2) 蒙特卡洛：估计量的方差比「常数」本身还大
      t       样本量          差值均值         跨 θ 的跨度          相对跨度
    0.5     20000        2.2928          0.1437         6.27%


    0.5    200000        2.2855          0.0698         3.05%
       → 样本量 ×10，散布变为原来的 0.49 倍（理想 MC 应为 0.32）
    0.9     20000       -0.0036          0.1260      3508.89%


    0.9    200000        0.0001          0.0693     47966.94%
       → 样本量 ×10，散布变为原来的 0.55 倍（理想 MC 应为 0.32）

  ⇒ 结论见表：t=0.9 时常数 C 本身接近 0（重尾路径下,u_t^{marg}
     含 1/(1−t) 因子），蒙特卡洛把它完全淹掉 —— 验证「只差一个常数」
     必须用确定性求积，判据是残差是否随网格收敛。



## D. velocity / score / denoiser / noise：四个名字，一个解

Prop 1 说 $u_t$ 与 $\nabla\log p_t$ 互为仿射重组；Remark 16 进一步说，
任何能反解出 $\mathbb E[z\mid x]$ 的量都行。于是四个参数化
（velocity $u_t$、score $s_t$、denoiser $D_t$、噪声 $\epsilon$）**最优解完全相同**，
差别只在损失函数里那条形如 $a_t^2$ 的**有效时间权重**。

这一组顺带给出两个工程结论：

1. VP 路径下 $a_t\equiv-\gamma/2$ 与 $t$ 无关 ⇒ CFM 与去噪 score matching 只差全局常数；
   CondOT 下 $a_t^2$ 跨越 4 个数量级 ⇒ 两者的时间权重天差地别。
2. 把 velocity 网络当 score 用时必须写成 $s_\theta=(v_\theta-b_tx)/a_t$，
   **不能**写成 $s_\theta=v_\theta/b_t$——丢掉 $b_tx$ 那项的代价见下表。

In [5]:
n_ok = L.exp_D(rng)

D | velocity / score / denoiser / noise：换的是参数化，不是解

  表 D1｜CondOT 路径：四个参数化之间来回换算的最大绝对误差
       t         a_t       b_t        a_t² (CFM/DSM 权重比)       u→s→u       u→D→u       u→ε→u
    0.05     19.0000    20.000                   361.000    2.13e-14    0.00e+00    1.38e-14
    0.20      4.0000     5.000                    16.000    3.55e-15    0.00e+00    4.44e-15
    0.35      1.8571     2.857                     3.449    2.66e-15    0.00e+00    3.55e-15
    0.50      1.0000     2.000                     1.000    1.78e-15    0.00e+00    1.78e-15
    0.65      0.5385     1.538                     0.290    3.55e-15    0.00e+00    3.55e-15
    0.80      0.2500     1.250                     0.062    3.55e-15    0.00e+00    7.11e-15
    0.95      0.0526     1.053                     0.003    7.11e-15    0.00e+00    2.84e-14
    [ok]   四种参数化互为仿射重组（误差 < 1e-10） worst 2.84e-14

  表 D2｜换一条路径，有效时间权重完全不同
       t   CondOT  a_t     trig  a_t    VP(γ=1)  a_t
    0.10        9.0000        9.9176    

## E. Thm 17：训练完之后，$\sigma_t$ 是一个免费的旋钮

$$dX_t=\Big[u_t(X_t)+\frac{\sigma_t^2}{2}\nabla\log p_t(X_t)\Big]dt+\sigma_t\,dW_t
\ \Longrightarrow\ X_t\sim p_t\quad(\forall\,\sigma_t\ge0).$$

也就是说：ODE 采样（$\sigma\equiv0$）和任意噪声水平的 SDE 采样，边际分布一样。
这张表用 Euler–Maruyama 把这条结论跑出来——**一直到 $\sigma$ 大到步长撑不住为止**。

讲义特别强调：**“最优 $\sigma_t$”是不完美模型 + 有限算力的产物**，
不是连续极限下动力学的性质。最后一行的 $\sigma=2.0$ 就是这句话的实测版本。

In [6]:
n_ok = L.exp_E(rng)

E | Thm 17：同一个 u_t，配任意 σ_t，终点分布不变



  噪底：两组各 60000 个「直接从 p_data 采」的样本之间的 W1 = 0.0151（这是下面几行能分辨的最小差异）
  模拟：Euler–Maruyama，400 步，dX = [u_t + σ²/2·∇log p_t] dt + σ dW

       σ_t     W1 到 p_data        终点均值         终点标准差


       0.0          0.0675     -0.7484        2.0327


       0.3          0.0479     -0.7196        2.0466


       0.6          0.0581     -0.7360        2.0395


       1.0          0.0555     -0.7193        2.0454


       2.0          0.1291     -0.7290        2.0516
    [ok]   σ ∈ [0,1] 四档终点 W1 极差 < 0.05 极差 0.0196
  → 前四档极差 0.0196，而 σ=2.0 那一档明显变差：
    Euler–Maruyama 的步长已经不够，模拟误差压过了理论不变性。
    这正是讲义说的：最优 σ 是「模型不完美 + 算力有限」的产物。



## F. Classifier-free guidance：从 Bayes 到“锐化”

三步推完：

1. Bayes 分解：$\nabla\log p_t(x\mid y)=\nabla\log p_t(x)+\nabla\log p_t(y\mid x)$，
   代入 Prop 1 得 $u_t(x\mid y)=u_t(x)+a_t\nabla\log p_t(y\mid x)$（classifier guidance）。
2. 把第二项放大 $w$ 倍，再把 $\nabla\log p_t(y\mid x)$ 反解成两个场的差，得到
   $\tilde u_t(x\mid y)=(1-w)u_t(x\mid\emptyset)+wu_t(x\mid y)$（CFG）。
3. 于是 CFG 场恰好是**倾斜分布** $\tilde p_t(x)\propto p_t(x)\,p_t(y\mid x)^w$ 的 Prop-1 速度场。

第 3 步是理解 CFG 的关键：当 $w=1$ 时 $\tilde p=p(\cdot\mid y)$，推断正确；
$w>1$ 时它采的是被 $p_t(y\mid x)^w$ 锐化过的分布——**CFG 是启发式，不是后验推断**。

In [7]:
n_ok = L.exp_F(rng)

F | Classifier-free guidance：Bayes → 放大 → (1−w)u(·|∅) + w u(·|y)

  表 F1｜两条恒等式（整条 x 网格逐点核对）
       t    u(x|y) = u(x) + a_t∇log p(y|x)   CFG = classifier guidance
    0.20                          1.15e-14                    8.26e-14
    0.50                          5.33e-15                    8.26e-14
    0.80                          7.11e-15                    8.26e-14
    [ok]   Bayes 分解出的引导场恒等式成立 1.15e-14
    [ok]   CFG 组合式 ≡ classifier guidance 放大式 8.26e-14

  表 F2｜w>1 之后，它采的是「倾斜分布」 p̃_t(x) ∝ p_t(x)·p_t(y|x)^w
       w           ∇log p̃ 重构的场与原 CFG 场的误差        终点均值        y=1 区域占比


     1.0                          1.07e-14      1.4949           0.499


     2.0                          1.60e-14      1.8633           0.585


     3.0                          2.84e-14      2.2288           0.673


     5.0                          4.80e-14      2.8575           0.824


     8.0                          8.17e-14      3.4189           0.956
    [ok]   CFG 场 = 倾斜分布的 Prop-1 速度场（逐点） 8.17e-14

  ⇒ w=1 时 p̃=p(·|y)，分布正确；w>1 就不再是 p_data(·|y)，
     而是把它按 p_t(y|x)^w 锐化——所以 CFG 是启发式，不是推断。



## G. 端到端：不借 torch，用 numpy 手训两个头

最后把整条流水线跑一遍：同一个数据、同一套超参，训练

* **velocity 头**：回归 $u_t(x\mid z)=A_tz+B_tx$（CondOT）
* **$\epsilon$ 头**：回归噪声 $\epsilon$（DDPM 形态），再用 $z=(x-\beta\epsilon)/\alpha$ 换回 velocity

如果「四个参数化同解」是真的，两个头换算回 velocity 之后应该给出同一个函数。
真的差别在**换算里的 $1/\alpha_t$**：$t$ 越小，同样的 $\epsilon$ 预测误差被放得越大——
这才是 CondOT 下工业界一律用 velocity 头的理由（不是表达力更强）。

In [8]:
n_ok = L.exp_G()

G | 端到端：numpy 手训 velocity 头与 ε 头，看它们是不是同一个解

  同一个 p_data（5 个原子，d=2），同一份超参（隐藏层 256，Adam lr=3e-3，batch=256）



  表 G1｜训练过程的目标 MSE（每 1/8 进度，用 40×2048 个样本估计）
   velocity: 2.7075  2.3467  1.8986  1.8572  1.8215  1.7931  1.7604  1.7220
    epsilon: 0.5881  0.4489  0.3734  0.3435  0.3211  0.3084  0.3031  0.2969

  表 G2｜两个头换算回 velocity 后与解析边际场 u_t 的 RMSE（越小越好）
       t          目标噪声    velocity 头           ε 头    ε/velocity         两个头之差
    0.05         2.214         0.452         5.484         12.13         5.400


    0.20         2.449         0.320         1.215          3.80         1.090
    0.50         2.023         0.447         0.617          1.38         0.512
    0.80         0.128         0.850         0.687          0.81         0.372


    0.95         0.000         0.948         0.845          0.89         0.320
    [ok]   两个头换算回 velocity 后彼此接近（相对 1/t 放大前的量级） dhead 0.320

  ⇒ 两个头收敛到同一个函数（差的那一列是同一个量级），真正的差别是
     ε→u 换算里那个 1/α_t：t 越小，同样的 ε 预测误差被放得越大，
     t=0.05 处 ε 头的 RMSE 已经是 velocity 头的 12.13 倍。
     这就是 CondOT 下工业界一律用 velocity 头的数值理由 —— 不是表达力。



## 结论速记

| 结论 | 实测 |
| :--- | :--- |
| 向量场 / score / denoiser 逐点相等 | 误差 $\le 5\times10^{-14}$，5 条路径一致 |
| 连续性方程是路径无关的契约 | 相对残差 $\le 6\times10^{-15}$ |
| 裸 $u_t$ + 噪声 ≠ 同一条 $p_t$ | $\sigma=0.8$ 时 FPE 残差 $2.43$ |
| 加上 $\frac{\sigma^2}{2}\nabla\log p_t$ 之后 | 残差回到 $5\times10^{-14}$ |
| $\mathcal L_{\rm FM}-\mathcal L_{\rm CFM}$ 与 $\theta$ 无关 | 求积相对跨度 $\le 10^{-12}$；MC 完全淹没 |
| 四种参数化同解 | 来回换算误差 $\le 2\times10^{-14}$ |
| $\sigma_t$ 任意可选 | $\sigma\in[0,1]$ 四档终点 W1 极差 $0.02$ |
| CFG ≡ 倾斜分布的真实速度场 | 逐点误差 $\le 8\times10^{-14}$ |
| velocity 头 vs $\epsilon$ 头 | $t=0.05$ 处后者 RMSE 高一个数量级 |

想复跑：`python3 fm_principles_lab.py`（默认 A–F；加参数可选组，如 `fm_principles_lab.py C E`，
所有输出已固化在 `results/stdout.txt`）。